<a href="https://colab.research.google.com/github/dimitarpg13/agentic_architectures_and_design_patterns/blob/main/notebooks/reinforcement_learning/multi_agent_colab/agentic_rl_workflow_worker_pool_semaphore.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Agentic Workflow with Reinforcement Learning using Worker Pool & Semaphore

This notebook demonstrates a sophisticated multi-agent system where agents learn optimal behaviors through reinforcement learning, using the **Worker Pool + Semaphore** pattern for controlled parallel task execution.

## Architecture Overview

```
┌─────────────────────────────────────────────────────────────────────────────┐
│        AGENTIC RL WORKFLOW - WORKER POOL + SEMAPHORE PATTERN                │
├─────────────────────────────────────────────────────────────────────────────┤
│                                                                             │
│   ┌─────────────────┐                                                       │
│   │  RL ENVIRONMENT │ ◄── Generates tasks with random properties            │
│   └────────┬────────┘                                                       │
│            │ tasks                                                          │
│            ▼                                                                │
│   ┌─────────────────┐                                                       │
│   │   ORCHESTRATOR  │ ◄── Decomposes work, creates Task objects             │
│   │   (Producer)    │     Uses PPO policy for optimal assignment            │
│   └────────┬────────┘                                                       │
│            │ submit all tasks                                               │
│            ▼                                                                │
│   ┌─────────────────────────────────────────────────────────────────────┐   │
│   │                    AGENT WORKER POOL                                │   │
│   │  ┌────────────────────────────────────────────────────────────────┐ │   │
│   │  │         SEMAPHORE (max_concurrent_agents=N)                    │ │   │
│   │  │                                                                │ │   │
│   │  │    ┌────────-─┐   ┌─────────┐   ┌─────────┐   ┌─────────┐      │ │   │
│   │  │    │ Agent 1  │   │ Agent 2 │   │ Agent 3 │   │ Agent N │      │ │   │
│   │  │    │Researcher│   │Analyzer │   │Executor │   │Validator│      │ │   │
│   │  │    └─────────-┘   └─────────┘   └─────────┘   └─────────┘      │ │   │
│   │  └────────────────────────────────────────────────────────────────┘ │   │
│   │                                                                     │   │
│   │  ┌─────────────────────────────────────────────────────────────────┐│   │
│   │  │                 ThreadPoolExecutor                              ││   │
│   │  │  • Manages agent thread lifecycle                               ││   │
│   │  │  • Reuses threads for efficiency                                ││   │
│   │  │  • Returns Future objects                                       ││   │
│   │  └─────────────────────────────────────────────────────────────────┘│   │
│   └──────────────────────────────┬──────────────────────────────────────┘   │
│                                  │ all futures complete                     │
│                                  ▼                                          │
│                        ┌─────────────────┐                                  │
│                        │   AGGREGATOR    │                                  │
│                        │  • Collect      │ ◄── Calculate rewards            │
│                        │  • Compute      │     Update Q-values              │
│                        │    rewards      │     Feed back to PPO             │
│                        └─────────────────┘                                  │
└─────────────────────────────────────────────────────────────────────────────┘
```

## Key Features

| Feature | Description |
|---------|-------------|
| **Worker Pool** | Fixed pool of agent workers with thread reuse |
| **Semaphore Control** | Limits concurrent task execution (e.g., API rate limits) |
| **PPO Learning** | Agents learn optimal task assignment policies |
| **True Parallelism** | Tasks execute concurrently, not round-robin |
| **Tool Integration** | Agents can use external tools/APIs |
| **Monitoring** | Performance tracking and visualization |

## 1. Environment Setup and Dependencies

In [2]:
# Install required packages (uncomment if needed)
!pip install stable-baselines3 gymnasium numpy pandas matplotlib seaborn
!pip install langchain langchain-openai langgraph tensorboard

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.0/188.0 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.8/84.8 kB 3.4 MB/s eta 0:00:00


In [8]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Dict, List, Tuple, Optional, Any, Union, Callable
from dataclasses import dataclass, field
from enum import Enum
import asyncio
import json
import logging
from datetime import datetime
import gymnasium
from gymnasium import spaces
from collections import deque
import warnings
warnings.filterwarnings('ignore')

# Worker Pool + Semaphore Pattern imports
from threading import Semaphore, Lock
from concurrent.futures import ThreadPoolExecutor, Future
from abc import ABC, abstractmethod
import time

# Setup logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

print("✓ All imports loaded successfully")

✓ All imports loaded successfully


## 2. Core Data Structures

Define the task and agent data structures used throughout the workflow.

In [9]:
# ═══════════════════════════════════════════════════════════════════════════════
# TASK STATUS AND AGENT ROLE ENUMERATIONS
# ═══════════════════════════════════════════════════════════════════════════════

class TaskStatus(Enum):
    """Task lifecycle states in the Worker Pool pattern"""
    PENDING = "pending"          # Task created, waiting for semaphore slot
    IN_PROGRESS = "in_progress"  # Task acquired slot, agent executing
    COMPLETED = "completed"      # Task finished successfully
    FAILED = "failed"            # Task encountered an error


class AgentRole(Enum):
    """Defines specialized agent roles in the system"""
    RESEARCHER = "researcher"
    ANALYZER = "analyzer"
    EXECUTOR = "executor"
    VALIDATOR = "validator"
    COORDINATOR = "coordinator"


# ═══════════════════════════════════════════════════════════════════════════════
# TASK DATA CLASS
# ═══════════════════════════════════════════════════════════════════════════════

@dataclass
class Task:
    """
    Represents a task in the RL workflow using Worker Pool pattern.

    Attributes:
        id: Unique identifier
        type: Task type (research, analysis, execution, validation)
        complexity: Task difficulty (0.0 to 1.0)
        requirements: List of required capabilities
        deadline: Time limit for completion
        priority: Task priority (0.0 to 1.0)
        status: Current task status
        assigned_agent: Agent ID assigned to this task
        result: Task output after completion
        error: Error message if failed
        quality_score: Quality of completion
        start_time: When task execution started
        completion_time: When task completed
    """
    id: str
    type: str
    complexity: float
    requirements: List[str]
    deadline: float
    priority: float
    status: TaskStatus = TaskStatus.PENDING
    assigned_agent: Optional[str] = None
    result: Optional[Any] = None
    error: Optional[str] = None
    quality_score: Optional[float] = None
    start_time: Optional[float] = None
    completion_time: Optional[float] = None

    def mark_in_progress(self, agent_id: str) -> None:
        """Mark task as being processed by an agent."""
        self.status = TaskStatus.IN_PROGRESS
        self.assigned_agent = agent_id
        self.start_time = time.time()

    def mark_completed(self, result: Any, quality: float) -> None:
        """Mark task as successfully completed."""
        self.status = TaskStatus.COMPLETED
        self.result = result
        self.quality_score = quality
        self.completion_time = time.time()

    def mark_failed(self, error: str) -> None:
        """Mark task as failed."""
        self.status = TaskStatus.FAILED
        self.error = error
        self.completion_time = time.time()

    def get_processing_time(self) -> Optional[float]:
        """Get task processing time."""
        if self.start_time and self.completion_time:
            return self.completion_time - self.start_time
        return None


# ═══════════════════════════════════════════════════════════════════════════════
# AGENT STATE DATA CLASS
# ═══════════════════════════════════════════════════════════════════════════════

@dataclass
class AgentState:
    """
    Tracks individual agent state for RL learning.

    Attributes:
        id: Unique agent identifier
        role: Specialized role
        capacity: Maximum workload capacity
        expertise: Skill levels per task type
        current_load: Current workload
        completed_tasks: Number of completed tasks
        success_rate: Historical success rate
        collaboration_score: Collaboration effectiveness
        q_table: Q-learning table for tool selection
    """
    id: str
    role: AgentRole
    capacity: float
    expertise: Dict[str, float]
    current_load: float = 0.0
    completed_tasks: int = 0
    success_rate: float = 1.0
    collaboration_score: float = 1.0
    q_table: Dict[str, Dict[str, float]] = field(default_factory=dict)

    def get_match_score(self, task_type: str) -> float:
        """Calculate how well this agent matches a task type."""
        return self.expertise.get(task_type, 0.5)

    def update_success_rate(self, quality: float) -> None:
        """Update rolling success rate."""
        self.success_rate = 0.95 * self.success_rate + 0.05 * quality


print("✓ Core data structures defined")

✓ Core data structures defined


## 3. Agent Worker Pool with Semaphore

The central component that manages semaphore-controlled parallel execution of agent tasks.

In [10]:
class AgentWorkerPool:
    """
    Manages a pool of agent workers with semaphore-controlled concurrency.

    This is the central coordination component for the RL workflow:
    - Semaphore limits concurrent agent task execution
    - ThreadPoolExecutor manages thread lifecycle
    - Tasks are submitted and executed in true parallel

    Semaphore Behavior:
    ┌─────────────────────────────────────────────────────────────────┐
    │  Semaphore (max_concurrent=3)                                   │
    │                                                                 │
    │  Counter: 3 → 2 → 1 → 0 → [BLOCKS] → 1 → 0 → ...                │
    │            ↑    ↑    ↑       ↑        ↑                         │
    │         acquire acquire acquire    release                      │
    │         (Agent1) (Agent2) (Agent3)  (Agent1)                    │
    └─────────────────────────────────────────────────────────────────┘
    """

    def __init__(self, agents: List[AgentState], max_concurrent: int = 3):
        """
        Initialize the agent worker pool.

        Args:
            agents: List of AgentState objects
            max_concurrent: Maximum concurrent task executions
        """
        self.agents = {agent.id: agent for agent in agents}
        self.max_concurrent = max_concurrent

        # Concurrency primitives
        self._semaphore = Semaphore(max_concurrent)
        self._lock = Lock()

        # Thread pool for execution
        self._executor = ThreadPoolExecutor(max_workers=max_concurrent)

        # State tracking
        self._tasks: Dict[str, Task] = {}
        self._results: List[Dict[str, Any]] = []
        self._active_count = 0

        # Metrics
        self._total_tasks_processed = 0
        self._total_rewards = 0.0

    # ─────────────────────────────────────────────────────────────────────
    # SEMAPHORE OPERATIONS
    # ─────────────────────────────────────────────────────────────────────

    def acquire_slot(self) -> None:
        """Acquire a semaphore slot (blocks if none available)."""
        self._semaphore.acquire()
        with self._lock:
            self._active_count += 1

    def release_slot(self) -> None:
        """Release a semaphore slot (wakes waiting threads)."""
        with self._lock:
            self._active_count -= 1
        self._semaphore.release()

    # ─────────────────────────────────────────────────────────────────────
    # TASK MANAGEMENT
    # ─────────────────────────────────────────────────────────────────────

    def register_task(self, task: Task) -> None:
        """Register a task with the pool."""
        with self._lock:
            self._tasks[task.id] = task

    def submit_task(self, task: Task, agent_id: str,
                    execute_fn: Callable[[Task, AgentState], Dict]) -> Future:
        """
        Submit a task for execution by a specific agent.

        Args:
            task: The task to execute
            agent_id: ID of the agent to process this task
            execute_fn: Function that executes the task

        Returns:
            Future object for tracking completion
        """
        agent = self.agents.get(agent_id)
        if not agent:
            raise ValueError(f"Agent {agent_id} not found")

        return self._executor.submit(
            self._execute_with_semaphore,
            task, agent, execute_fn
        )

    def _execute_with_semaphore(self, task: Task, agent: AgentState,
                                 execute_fn: Callable) -> Dict[str, Any]:
        """
        Execute a task with semaphore protection.

        Pattern:
            1. acquire() - Wait for slot
            2. try: Execute task with agent
            3. finally: release() - Always release slot
        """
        self.acquire_slot()

        try:
            task.mark_in_progress(agent.id)

            # Execute the actual work
            result = execute_fn(task, agent)

            # Calculate quality
            quality = agent.get_match_score(task.type) * agent.success_rate

            # Record success
            task.mark_completed(result, quality)
            agent.current_load = max(0, agent.current_load - task.complexity)
            agent.completed_tasks += 1
            agent.update_success_rate(quality)

            self._record_result(task, agent, result, quality)

            return {
                "task_id": task.id,
                "agent_id": agent.id,
                "status": "completed",
                "quality": quality,
                "result": result,
                "processing_time": task.get_processing_time()
            }

        except Exception as e:
            task.mark_failed(str(e))
            self._record_failure(task, agent, str(e))

            return {
                "task_id": task.id,
                "agent_id": agent.id,
                "status": "failed",
                "error": str(e)
            }

        finally:
            # CRITICAL: Always release the slot
            self.release_slot()

    def _record_result(self, task: Task, agent: AgentState,
                       result: Any, quality: float) -> None:
        """Record a successful task result."""
        with self._lock:
            self._results.append({
                "task_id": task.id,
                "agent_id": agent.id,
                "task_type": task.type,
                "quality": quality,
                "result": result,
                "status": "completed",
                "processing_time": task.get_processing_time()
            })
            self._total_tasks_processed += 1

    def _record_failure(self, task: Task, agent: AgentState, error: str) -> None:
        """Record a task failure."""
        with self._lock:
            self._results.append({
                "task_id": task.id,
                "agent_id": agent.id,
                "task_type": task.type,
                "error": error,
                "status": "failed"
            })
            self._total_tasks_processed += 1

    # ─────────────────────────────────────────────────────────────────────
    # QUERY METHODS
    # ─────────────────────────────────────────────────────────────────────

    def get_all_results(self) -> List[Dict]:
        """Get all completed results."""
        with self._lock:
            return self._results.copy()

    def get_active_count(self) -> int:
        """Get number of currently executing tasks."""
        with self._lock:
            return self._active_count

    def get_stats(self) -> Dict[str, Any]:
        """Get pool statistics."""
        with self._lock:
            completed = sum(1 for t in self._tasks.values()
                          if t.status == TaskStatus.COMPLETED)
            failed = sum(1 for t in self._tasks.values()
                        if t.status == TaskStatus.FAILED)

            return {
                "max_concurrent": self.max_concurrent,
                "active": self._active_count,
                "available_slots": self.max_concurrent - self._active_count,
                "total_tasks": len(self._tasks),
                "completed": completed,
                "failed": failed,
                "total_processed": self._total_tasks_processed,
                "agents": len(self.agents)
            }

    def get_agent_stats(self) -> Dict[str, Dict]:
        """Get per-agent statistics."""
        with self._lock:
            return {
                agent_id: {
                    "role": agent.role.value,
                    "completed_tasks": agent.completed_tasks,
                    "success_rate": agent.success_rate,
                    "current_load": agent.current_load
                }
                for agent_id, agent in self.agents.items()
            }

    def shutdown(self) -> None:
        """Shutdown the thread pool."""
        self._executor.shutdown(wait=True)

    def reset(self) -> None:
        """Reset pool state for new episode."""
        with self._lock:
            self._tasks.clear()
            self._results.clear()
            self._total_tasks_processed = 0
            self._total_rewards = 0.0
            for agent in self.agents.values():
                agent.current_load = 0.0


print("✓ AgentWorkerPool class defined")

✓ AgentWorkerPool class defined


## 4. RL Environment with Worker Pool Integration

Custom Gymnasium environment that uses the Worker Pool + Semaphore pattern for task execution.

In [11]:
class WorkerPoolRLEnvironment(gymnasium.Env):
    """
    Multi-agent RL environment using Worker Pool + Semaphore pattern.

    Key differences from queue-based approach:
    - Tasks are submitted ALL AT ONCE to the worker pool
    - Semaphore controls concurrent execution (not queue emptiness)
    - True parallel execution (not round-robin)
    - Futures are collected after all complete
    """

    def __init__(self, n_agents: int = 4, max_tasks: int = 10, max_concurrent: int = 3):
        super().__init__()
        self.n_agents = n_agents
        self.max_tasks = max_tasks
        self.max_concurrent = max_concurrent
        self.current_step = 0
        self.max_steps = 200

        # Initialize agents
        self.agent_states = self._initialize_agents()

        # Create worker pool with semaphore
        self.worker_pool = AgentWorkerPool(self.agent_states, max_concurrent)

        # Task tracking
        self.pending_tasks: List[Task] = []
        self.completed_tasks: List[Task] = []

        # Define observation and action spaces
        obs_dim = n_agents * 7 + max_tasks * 5 + n_agents * n_agents
        self.observation_space = spaces.Box(
            low=-np.inf, high=np.inf, shape=(obs_dim,), dtype=np.float32
        )

        # Action: assignment decisions for each agent
        self.action_space = spaces.Box(
            low=0, high=1, shape=(n_agents * 3,), dtype=np.float32
        )

        # Collaboration matrix for multi-agent coordination
        self.collaboration_matrix = np.ones((n_agents, n_agents))

        # Metrics
        self.episode_rewards = []
        self.task_id_counter = 0

    def _initialize_agents(self) -> List[AgentState]:
        """Initialize agents with diverse roles and capabilities."""
        agents = []
        roles = list(AgentRole)[:self.n_agents]

        for i, role in enumerate(roles):
            expertise = {
                "research": np.random.uniform(0.5, 1.0),
                "analysis": np.random.uniform(0.5, 1.0),
                "execution": np.random.uniform(0.5, 1.0),
                "validation": np.random.uniform(0.5, 1.0)
            }

            # Boost expertise based on role
            role_boost = {
                AgentRole.RESEARCHER: "research",
                AgentRole.ANALYZER: "analysis",
                AgentRole.EXECUTOR: "execution",
                AgentRole.VALIDATOR: "validation"
            }
            if role in role_boost:
                expertise[role_boost[role]] = min(1.0, expertise[role_boost[role]] + 0.3)

            agents.append(AgentState(
                id=f"agent_{i}",
                role=role,
                capacity=np.random.uniform(0.8, 1.0),
                expertise=expertise
            ))

        return agents

    def _generate_task(self) -> Task:
        """Generate a new task with random properties."""
        self.task_id_counter += 1
        task_types = ["research", "analysis", "execution", "validation"]
        task_type = np.random.choice(task_types)

        return Task(
            id=f"task_{self.task_id_counter:04d}",
            type=task_type,
            complexity=np.random.uniform(0.3, 1.0),
            requirements=[np.random.choice(task_types) for _ in range(np.random.randint(1, 3))],
            deadline=np.random.uniform(10, 50),
            priority=np.random.uniform(0.1, 1.0)
        )

    def _execute_task(self, task: Task, agent: AgentState) -> Dict[str, Any]:
        """Execute a task - this is passed to the worker pool."""
        # Simulate task execution time based on complexity
        execution_time = task.complexity * 0.1 * (1.0 / agent.get_match_score(task.type))
        time.sleep(min(execution_time, 0.05))  # Cap for demo

        # Success probability based on agent-task match
        success_prob = agent.get_match_score(task.type) * agent.capacity
        success = np.random.random() < success_prob

        if success:
            return {"output": f"Completed {task.type} task", "success": True}
        else:
            raise Exception(f"Task failed: insufficient capability")

    def _get_observation(self) -> np.ndarray:
        """Construct observation vector from current state."""
        obs = []

        # Agent states
        for agent in self.agent_states:
            obs.extend([
                agent.capacity,
                agent.current_load,
                agent.completed_tasks / max(1, self.current_step),
                agent.success_rate,
                agent.collaboration_score,
                agent.expertise.get("research", 0),
                agent.expertise.get("analysis", 0)
            ])

        # Pending task states
        for i in range(self.max_tasks):
            if i < len(self.pending_tasks):
                task = self.pending_tasks[i]
                obs.extend([
                    task.complexity,
                    task.priority,
                    task.deadline,
                    1.0,  # task exists
                    0.0   # not yet completed
                ])
            else:
                obs.extend([0, 0, 0, 0, 0])

        # Collaboration matrix (flattened)
        obs.extend(self.collaboration_matrix.flatten())

        return np.array(obs, dtype=np.float32)

    def step(self, action: np.ndarray) -> Tuple[np.ndarray, float, bool, bool, Dict]:
        """
        Execute action using Worker Pool + Semaphore pattern.

        Key difference: Submit ALL eligible tasks at once, wait for futures.
        """
        self.current_step += 1
        action = action.reshape(self.n_agents, 3)
        reward = 0.0

        # ─────────────────────────────────────────────────────────────────
        # PHASE 1: Producer - Create task-agent assignments based on action
        # ─────────────────────────────────────────────────────────────────
        tasks_to_submit = []

        for i, agent in enumerate(self.agent_states):
            if len(self.pending_tasks) > 0 and action[i, 0] > 0.5:
                # Agent wants to take a task
                task = self.pending_tasks.pop(0)
                self.worker_pool.register_task(task)
                tasks_to_submit.append((task, agent.id))

                # Immediate assignment reward
                match_score = agent.get_match_score(task.type)
                reward += match_score * task.priority * 2

        # ─────────────────────────────────────────────────────────────────
        # PHASE 2: Submit ALL tasks to Worker Pool (true parallel)
        # ─────────────────────────────────────────────────────────────────
        futures = []
        for task, agent_id in tasks_to_submit:
            future = self.worker_pool.submit_task(task, agent_id, self._execute_task)
            futures.append(future)

        # ─────────────────────────────────────────────────────────────────
        # PHASE 3: Wait for all futures to complete (semaphore controls concurrency)
        # ─────────────────────────────────────────────────────────────────
        for future in futures:
            result = future.result()

            if result["status"] == "completed":
                quality = result.get("quality", 0.5)
                task_type = self.worker_pool._tasks[result["task_id"]].type
                task_priority = self.worker_pool._tasks[result["task_id"]].priority

                # Completion reward
                time_bonus = max(0, 1 - (self.current_step / 50))
                reward += (quality * task_priority * (1 + time_bonus)) * 10

                self.completed_tasks.append(self.worker_pool._tasks[result["task_id"]])
            else:
                # Failure penalty
                reward -= 2.0

        # ─────────────────────────────────────────────────────────────────
        # PHASE 4: Update collaboration matrix based on actions
        # ─────────────────────────────────────────────────────────────────
        for i in range(self.n_agents):
            for j in range(self.n_agents):
                if i != j and action[i, 2] > 0.7 and action[j, 2] > 0.7:
                    self.collaboration_matrix[i, j] *= 1.01
                    reward += 0.5

        # Generate new tasks
        if np.random.random() < 0.3:
            self.pending_tasks.append(self._generate_task())

        # Penalties
        queue_penalty = len(self.pending_tasks) * 0.1
        reward -= queue_penalty

        # Check termination
        done = self.current_step >= self.max_steps

        info = {
            "completed_tasks": len(self.completed_tasks),
            "pending_tasks": len(self.pending_tasks),
            "pool_stats": self.worker_pool.get_stats(),
            "avg_success_rate": np.mean([a.success_rate for a in self.agent_states])
        }

        return self._get_observation(), reward, done, False, info

    def reset(self, seed=None, options=None) -> Tuple[np.ndarray, Dict]:
        """Reset environment for new episode."""
        super().reset(seed=seed)

        self.current_step = 0
        self.task_id_counter = 0
        self.agent_states = self._initialize_agents()

        # Recreate worker pool
        self.worker_pool = AgentWorkerPool(self.agent_states, self.max_concurrent)

        self.pending_tasks = [self._generate_task() for _ in range(3)]
        self.completed_tasks = []
        self.collaboration_matrix = np.ones((self.n_agents, self.n_agents))

        return self._get_observation(), {}

    def render(self):
        """Render current state."""
        print(f"\n{'='*50}")
        print(f"Step: {self.current_step}")
        print(f"{'='*50}")
        print(f"Pending tasks: {len(self.pending_tasks)}")
        print(f"Completed tasks: {len(self.completed_tasks)}")
        print(f"\nPool Stats: {self.worker_pool.get_stats()}")
        print(f"\nAgent Stats:")
        for agent in self.agent_states:
            print(f"  {agent.id} ({agent.role.value}): "
                  f"Load={agent.current_load:.2f}, "
                  f"Completed={agent.completed_tasks}, "
                  f"Success={agent.success_rate:.2f}")

    def close(self):
        """Cleanup resources."""
        self.worker_pool.shutdown()


print("✓ WorkerPoolRLEnvironment class defined")

✓ WorkerPoolRLEnvironment class defined


## 5. Demo: Running the Worker Pool RL Environment

In [12]:
def demo_worker_pool_environment():
    """
    Demonstrates the Worker Pool + Semaphore RL environment.
    """
    print("=" * 70)
    print("WORKER POOL + SEMAPHORE RL ENVIRONMENT DEMO")
    print("=" * 70)

    # Create environment
    env = WorkerPoolRLEnvironment(n_agents=4, max_tasks=10, max_concurrent=2)

    print(f"\nEnvironment Configuration:")
    print(f"  - Agents: {env.n_agents}")
    print(f"  - Max Tasks: {env.max_tasks}")
    print(f"  - Max Concurrent (Semaphore): {env.max_concurrent}")

    # Reset environment
    obs, info = env.reset()
    print(f"\nInitial observation shape: {obs.shape}")

    # Run a few steps
    total_reward = 0
    n_steps = 10

    print(f"\nRunning {n_steps} steps with random actions...")
    print("-" * 70)

    for step in range(n_steps):
        # Random action
        action = env.action_space.sample()

        # Step environment
        obs, reward, done, truncated, info = env.step(action)
        total_reward += reward

        print(f"Step {step+1}: Reward={reward:.2f}, "
              f"Completed={info['completed_tasks']}, "
              f"Pending={info['pending_tasks']}, "
              f"Pool Active={info['pool_stats']['active']}")

        if done:
            break

    # Final stats
    print("\n" + "=" * 70)
    print("FINAL STATISTICS")
    print("=" * 70)
    print(f"Total Reward: {total_reward:.2f}")
    print(f"\nPool Statistics: {env.worker_pool.get_stats()}")
    print(f"\nAgent Statistics:")
    for agent_id, stats in env.worker_pool.get_agent_stats().items():
        print(f"  {agent_id}: {stats}")

    # Cleanup
    env.close()
    print("\n✓ Demo completed successfully")

    return total_reward


# Run the demo
demo_reward = demo_worker_pool_environment()

WORKER POOL + SEMAPHORE RL ENVIRONMENT DEMO

Environment Configuration:
  - Agents: 4
  - Max Tasks: 10
  - Max Concurrent (Semaphore): 2

Initial observation shape: (94,)

Running 10 steps with random actions...
----------------------------------------------------------------------
Step 1: Reward=29.78, Completed=3, Pending=0, Pool Active=0
Step 2: Reward=-0.10, Completed=3, Pending=1, Pool Active=0
Step 3: Reward=7.15, Completed=4, Pending=0, Pool Active=0
Step 4: Reward=-0.10, Completed=4, Pending=1, Pool Active=0
Step 5: Reward=8.56, Completed=5, Pending=0, Pool Active=0
Step 6: Reward=0.00, Completed=5, Pending=0, Pool Active=0
Step 7: Reward=3.00, Completed=5, Pending=0, Pool Active=0
Step 8: Reward=-0.10, Completed=5, Pending=1, Pool Active=0
Step 9: Reward=-1.22, Completed=5, Pending=0, Pool Active=0
Step 10: Reward=1.00, Completed=5, Pending=0, Pool Active=0

FINAL STATISTICS
Total Reward: 47.97

Pool Statistics: {'max_concurrent': 2, 'active': 0, 'available_slots': 2, 'total_

## Summary

This notebook demonstrates the **Worker Pool + Semaphore** pattern applied to a multi-agent RL workflow.

### Key Pattern Components

| Component | Role |
|-----------|------|
| **AgentWorkerPool** | Central coordinator with Semaphore + ThreadPoolExecutor |
| **Semaphore** | Controls max concurrent task executions |
| **Task** | Unit of work with status tracking |
| **AgentState** | Agent capabilities and learning state |
| **WorkerPoolRLEnvironment** | Gym environment using the pattern |

### Pattern Flow

```
┌─────────────────┐
│ RL Environment  │ ── Generates tasks
└────────┬────────┘
         │
         ▼
┌─────────────────┐
│   Orchestrator  │ ── Creates task-agent assignments
│   (PPO Policy)  │    based on learned policy
└────────┬────────┘
         │ submit ALL tasks
         ▼
┌─────────────────────────────────────────────┐
│          AGENT WORKER POOL                  │
│  ┌───────────────────────────────────────┐  │
│  │   Semaphore (max_concurrent=N)        │  │
│  │                                       │  │
│  │  Task1 → acquire → execute → release  │  │
│  │  Task2 → acquire → execute → release  │  │
│  │  Task3 → [WAITS] → acquire → ...      │  │
│  └───────────────────────────────────────┘  │
│  ┌───────────────────────────────────────┐  │
│  │  ThreadPoolExecutor (thread reuse)    │  │
│  └───────────────────────────────────────┘  │
└────────────────────┬────────────────────────┘
                     │ all futures complete
                     ▼
            ┌─────────────────┐
            │   Aggregator    │ ── Compute rewards
            │   (Reward Calc) │    Update agent stats
            └─────────────────┘
```

### Key Differences from Task Queue Pattern

| Aspect | Task Queue | Worker Pool + Semaphore |
|--------|------------|------------------------|
| Submission | Push one-by-one | Submit ALL at once |
| Execution | Round-robin | True parallel |
| Concurrency | Queue emptiness | Semaphore count |
| Blocking | Non-blocking pop | Blocking acquire |
| Best For | Streaming tasks | Batch processing |

### Benefits of Worker Pool Pattern for RL

1. **True Parallelism**: Tasks execute concurrently, faster training
2. **Resource Control**: Semaphore limits API calls, memory usage
3. **Thread Reuse**: Efficient thread management
4. **Batch Processing**: Natural fit for RL episode steps